# Analiza streszczeń literatury młodzieżowej
## Przygotowanie środowiska

In [ ]:
python -m venv .venv
source .venv/bin/activate

In [ ]:
pip install pandas matplotlib nltk scikit-learn wordcloud morfeusz2

## Import bibliotek

In [ ]:
import os
import json
import string
import morfeusz2
import numpy as np
import pandas as pd
import operator as op
import itertools as it
from wordcloud import WordCloud
from matplotlib import pyplot as plt
from nltk import ngrams
from nltk.tokenize import word_tokenize
from nltk.corpus import PlaintextCorpusReader
from scipy.cluster.hierarchy import dendrogram
from sklearn.cluster import AgglomerativeClustering
from sklearn.decomposition import NMF, LatentDirichletAllocation
from sklearn.metrics.pairwise import cosine_similarity, euclidean_distances
from sklearn.feature_extraction.text import TfidfVectorizer, CountVectorizer

In [ ]:
pip freeze > requirements.txt

In [ ]:
import nltk
nltk.download('punkt_tab')
if "." not in nltk.data.path:
    nltk.data.path.append(".")

## Zdefiniowanie funkcji pomocniczych

In [ ]:
def print_dict(obj):
    print(json.dumps(obj, indent=4, ensure_ascii=False))

In [ ]:
def create_dir(path):
    if not os.path.exists(path):
        os.mkdir(path)

## Przygotowanie danych wejściowych
### Utworzenie korpusu dokumentów

In [ ]:
corpus_dir = "./literature"
corpus = PlaintextCorpusReader(corpus_dir, r'.*\.txt')
files_names = corpus.fileids()
titles = [file_name.removesuffix(".txt") for file_name in files_names]


In [ ]:
# Na linuksach będzie permission error w następnej komendzie, więc trzeba zmienić uprawnienia do katalogu ./literature i jego zawartości
# Z jakiegoś powodu nie działa to z komórki ipynb, więc trzeba to zrobić w terminalu.
chmod go-w ./literature/*

In [ ]:
documents_dict = {title: corpus.raw(file_name) for title, file_name in zip(titles, files_names)}
print_dict(documents_dict)

In [ ]:
documents_params = {}
for title, document in documents_dict.items():
    documents_params[title] = {
        "pre": len(word_tokenize(document, language="polish")),
    }
print_dict(documents_params)

### Utworzenie macierzy częstości (przed wstępnym przetwarzaniem)

In [ ]:
documents_df_pre = pd.DataFrame.from_dict(documents_dict, orient="index")
documents_df_pre.columns = ["content"]
display(documents_df_pre)

In [ ]:
cv = CountVectorizer()
tv = TfidfVectorizer()

In [ ]:
matrix_tf = cv.fit_transform(documents_df_pre["content"])
sparsity_tf = 1.0 - (matrix_tf.nnz / float(matrix_tf.shape[0] * matrix_tf.shape[1]))

display(pd.DataFrame(matrix_tf.toarray(), index=documents_df_pre.index, columns=cv.get_feature_names_out())) # Nie potrzebne, dodałem z ciekawości

In [ ]:
matrices_params = {
    "pre": {
        "words": matrix_tf.shape[1],
        "sparsity": sparsity_tf,
    }
}
print_dict(matrices_params)

### Wstępne przygotowanie dokumentów

In [ ]:
stopwords_file = open("./stopwords_pl.txt", "r", encoding="utf-8")
stopwords = stopwords_file.read().splitlines()
stopwords_file.close()
stopwords = stopwords[4:] # Usuwam pierwsze 4 linie, bo są komentarzami

In [ ]:
def lemmatize(text):
    lemmatizer = morfeusz2.Morfeusz()
    segments = it.groupby(lemmatizer.analyse(text), key=op.itemgetter(0, 1)) # Grupuję takie same segmenty, żeby wybrać najkrótszy opis (najbardziej ogólny) dla danego słowa
    def disambiguate(group):
        pairs = ((len(descr), lemma) for _, _, (_, lemma, descr, _, _, ) in group) # Tworzę pary (długość opisu, opis) dla danego słowa, znak "_" oznacza, że nie interesuje mnie dana wartość, mówi programowi, że nie będę jej używał
        _, lemma = min(pairs) # Wybieram opis o najmniejszej długości (najbardziej ogólny) dla danego słowa
        return lemma.split(":")[0] # Zwracam tylko samą formę podstawową (bez dodatkowych informacji po dwukropku)
    return " ".join(filter(str.isalpha, (disambiguate(group) for _, group in segments))) # Dla każdego segmentu wybieram jego formę podstawową, a następnie łączę je w jeden tekst, filtrując tylko te, które są alfabetami (czyli pomijam znaki interpunkcyjne i cyfry)

In [ ]:
for title, document in documents_dict.items():
    documents_dict[title] = document.lower()
    documents_dict[title] = "".join(filter(lambda char: char not in string.punctuation, documents_dict[title]))
    documents_dict[title] = lemmatize(documents_dict[title])
    documents_dict[title] = " ".join(filter(lambda word: word not in stopwords, word_tokenize(documents_dict[title], language="polish")))
print_dict(documents_dict)

In [ ]:
for title, document in documents_dict.items():
    documents_params[title]["post"] = len(word_tokenize(document, language="polish"))
print_dict(documents_params)

In [ ]:
documents_params_df = pd.DataFrame.from_dict(documents_params, orient="index")
documents_params_df.columns = ["pre", "post"]
documents_params_df["change"] = documents_params_df["post"] - documents_params_df["pre"]
documents_params_df["change_percent"] = (documents_params_df["change"] / documents_params_df["pre"]) * 100
display(documents_params_df)

### Utworzenie macierzy częstości (po wstępnym przetwarzaniu)

In [ ]:
documents_df_pre = pd.DataFrame.from_dict(documents_dict, orient="index")
documents_df_pre.columns = ["content"]
display(documents_df_pre)

In [ ]:
matrix_tf = cv.fit_transform(documents_df_pre["content"])
matrix_tfidf = tv.fit_transform(documents_df_pre["content"])
sparsity_tf = 1.0 - (matrix_tf.nnz / float(matrix_tf.shape[0] * matrix_tf.shape[1]))

display(pd.DataFrame(matrix_tf.toarray(), index=documents_df_pre.index, columns=cv.get_feature_names_out())) # Nie potrzebne, dodałem z ciekawości

In [ ]:
matrices_params["post"] = {

        "words": matrix_tfidf.shape[1],
        "sparsity": sparsity_tf,
}
print_dict(matrices_params)

In [ ]:
matrices_params_df = pd.DataFrame.from_dict(matrices_params, orient="index")
display(matrices_params_df)